
## GiftGeniusAgent — ваш умный агент для подбора подарков

### О проекте
Этот проект демонстрирует интеллектуального агента для подбора подарков на базе фреймворка HelloAgents.

### Об авторе
- Имя: Чжан Шаньци
- GitHub: @jack6249
- Дата: 2025-11-21


### Часть 1: Настройка окружения


In [ ]:
# Импорт библиотек и настройка параметров
from hello_agents import SimpleAgent, HelloAgentsLLM, ReflectionAgent, ToolRegistry
from hello_agents.tools import Tool, ToolParameter
from typing import Dict, Any, List
from tavily import TavilyClient
import os
import json
import re
import numpy as np 
from dotenv import load_dotenv
import asyncio
import nest_asyncio
from mcp.client.sse import sse_client
from mcp.client.session import ClientSession

load_dotenv()

# Параметры LLM
LLM_MODEL_ID = os.getenv("LLM_MODEL_ID")
LLM_API_KEY = os.getenv("LLM_API_KEY")
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
# Параметры Tavily
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY","")
# Параметры Baidu MCP
BAIDU_TOKEN = os.getenv("BAIDU_MCP_TOKEN","")
# Путь к входному JSON
INPUT_FILENAME = "data/test_cases.json"

# Настройка источника поиска
# Варианты: "tavily" (универсальный/зарубежный) или "baidu" (e-commerce/Китай)
os.environ["SEARCH_PROVIDER"] = "baidu" 

print("✅ Настройка окружения завершена")


### Часть 2: Определение инструментов


In [ ]:
# [Cell 2 финальная версия] Унифицированный инструмент поиска (Tavily и Baidu)
# Разрешаем Jupyter выполнять async-код
nest_asyncio.apply()

class BatchSearchTool(Tool):
    def __init__(self):
        super().__init__(
            name="batch_search",
            description="Унифицированный инструмент поиска с переключением Tavily и Baidu."
        )
        self.provider = os.environ.get("SEARCH_PROVIDER", "tavily").lower()

    def run(self, parameters: Any) -> str:
        return "Используйте Python-код и вызывайте search_raw напрямую для получения данных."

    def search_raw(self, query: str) -> List[Dict]:
        if self.provider == "baidu":
            return self._search_baidu(query)
        else:
            return self._search_tavily(query)

    # --- Движок A: Tavily ---
    def _search_tavily(self, query: str) -> List[Dict]:
        api_key = os.environ.get("TAVILY_API_KEY")
        if not api_key: return []
        print(f"    🚀 [Tavily] Поиск: {query} ...")
        try:
            tavily = TavilyClient(api_key=api_key)
            response = tavily.search(query, max_results=5, include_images=True)
            results = []
            if 'results' in response:
                for r in response['results']:
                    results.append({
                        "title": r['title'], "url": r['url'], "content": r['content'], 
                        "type": "text", "img": "" # У Tavily текст обычно без изображения
                    })
            if 'images' in response and response['images']:
                results.append({"images": response['images'][:3], "type": "image"})
            return results
        except Exception as e:
            print(f"      ⚠️ Ошибка Tavily: {e}")
            return []

    # --- Движок B: Baidu MCP ---
    def _search_baidu(self, query: str) -> List[Dict]:
        token = os.environ.get("BAIDU_MCP_TOKEN")
        if not token: return []
        print(f"    🐼 [Baidu Youxuan] Поиск: {query} ...")
        try:
            raw_json_str = asyncio.run(self._async_baidu_call(query, token))
            print(f"      🔍 Исходный JSON-ответ: {raw_json_str}")
            return self._parse_baidu_response(raw_json_str)
        except Exception as e:
            print(f"      ⚠️ Ошибка Baidu MCP: {e}")
            return []

    async def _async_baidu_call(self, query: str, token: str) -> str:
        sse_url = f"https://mcp-youxuan.baidu.com/mcp/sse?key={token}"
        async with sse_client(sse_url) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                result = await session.call_tool("goods_search", arguments={"query": query})
                return result.content[0].text if result.content else ""

    def _parse_baidu_response(self, json_str: str) -> List[Dict]:
        results = []; images = []
        try:
            data = json.loads(json_str)
            items = data if isinstance(data, list) else []
            
            for item in items[:5]:
                title = item.get("goodsName") or item.get("title") or "Неизвестный товар"
                price = item.get("price") or item.get("minPrice") or ""
                shop = item.get("shopName") or item.get("mall") or ""
                url = item.get("detailUrl") or item.get("url") or item.get("ori_url") or "#"
                img = item.get("imgUrl") or item.get("picUrl") or item.get("img")
                
                content = f"Цена: {price} юаней. Магазин: {shop}. Описание: {title}"
                
                # 📝【Исправление】Привязываем img прямо к результату типа text
                results.append({
                    "title": title, "url": url, "content": content, 
                    "type": "text", "img": img 
                })
                if img: images.append(img)
            
            if images: results.append({"images": images[:3], "type": "image"})
                
        except json.JSONDecodeError:
            print("      ⚠️ Baidu вернул данные не в формате JSON")
        return results

    def get_parameters(self):
        return [ToolParameter(name="query", type="string", description="Ключевое слово")]

tool_registry = ToolRegistry()
tool_registry.register_tool(BatchSearchTool())

print("✅ Унифицированный инструмент поиска загружен!")
print(f"Текущий режим: {'Baidu Youxuan (e-commerce)' if os.environ.get('SEARCH_PROVIDER') == 'baidu' else 'Tavily (универсальный)'}")

### Часть 3: Создание агентов



In [ ]:
# Инициализация большой языковой модели
llm = HelloAgentsLLM()

# --- 1. Стратег (Profiler) — поддержка многомерного профиля ---
PROFILER_PROMPT = """
Ты — «стратег по подаркам», эксперт в анализе MBTI и трендах потребительского рынка.
Твоя задача — на основе многомерного профиля пользователя составить 3 **максимально точных** поисковых запроса.

【⚠️ КРИТИЧЕСКОЕ ТРЕБОВАНИЕ ПО АКТУАЛЬНОСТИ】
Текущая дата считается **ноябрь 2025 года**.
1. **Запрет на устаревшее**: ни в коем случае не рекомендуй модели 2024 года или раньше (кроме вечной классики вроде виниловых пластинок).
2. **Шкала цен**: цены указаны в китайских юанях (CNY). Строго соблюдай диапазон бюджета, не выходи за его пределы.

Для качества рекомендаций используй следующие 【примеры хорошего мышления】:

# ## Пример 1
**Профиль пользователя**:
- Женщина, 26 лет, ISFP (Авантюрист), Телец
- Бюджет: 500–1000 юаней
- Повод: День святого Валентина
- Дополнительно: любит предметы с хорошей текстурой для дома
**Анализ стратега**:
ISFP ценит эстетику и сенсорный опыт, Телец любит осязаемое качество. На День святого Валентина нужна романтика.
**Стратегия поиска**:
1. 观夏 (To Summer) 昆仑煮雪 晶石香薰 (текстура и эстетика)
2. 野兽派 2025 情人节限定 睡衣礼盒 (комфорт, который любит Телец)
3. 富士 Instax mini Evo 拍立得 (запечатлеть моменты жизни)

# ## Пример 2
**Профиль пользователя**:
- Мужчина, 30 лет, INTJ (Архитектор), Дева
- Бюджет: более 1000 юаней
- Повод: День рождения
- Дополнительно: программист, любит порядок
**Анализ стратега**:
INTJ стремится к логике и эффективности, Дева любит чистоту и аккуратный рабочий стол.
**Стратегия поиска**:
1. Keychron Q1 Pro 机械键盘 铝坨坨 (инструмент для гика)
2. 明基 (BenQ) ScreenBar Halo 屏幕挂灯 (защита глаз и эстетика стола)
3. 赫曼米勒 (Herman Miller) 显示器支架 (эргономика)

# ## Пример 3
**Профиль пользователя**:
- Женщина, 20 лет, ENFP (Борец), Лев
- Бюджет: до 300 юаней
- Повод: Рождество
- Дополнительно: любит аниме, ita-bag (痛包)
**Анализ стратега**:
ENFP энергична и открыта, Лев любит яркие и броские вещи. Бюджет ограничен, но нужно много деталей.
**Стратегия поиска**:
1. 泡泡玛特 圣诞系列 盲盒整端 (праздничная атмосфера и аниме)
2. WEGO 痛包 镭射款 (ita-bag, яркий стиль для Льва)
3. Chiikawa 吉伊卡哇 圣诞公仔 (популярный аниме-IP)

---

**Текущая задача**:
Проанализируй 【текущий профиль пользователя】 ниже и составь стратегию поиска по глубине примеров выше.

【Текущий профиль пользователя】
{user_profile_text}

【Требования к ключевым словам】
1. **Конкретность**: формат `[бренд] + [название/серия] + [лимитка/атрибут]`.
2. **Без общих слов**: запрещены запросы вроде «подарок», «помада», «игрушка».
3. **Обязателен бренд**: подбирай бренд по бюджету (низкий — Miniso/泡泡玛特, высокий — Dior/索尼).

【Формат вывода】
Только 3 строки с ключевыми словами, по одному на строку. Без анализа и без нумерации.
"""

profiler_agent = SimpleAgent(
    llm=llm,
    name="Agent_Profiler",
    system_prompt=PROFILER_PROMPT
)

# ==============================================================================
# 2. Копирайтер (Pitcher) — создание продающих текстов
# ==============================================================================
PITCHER_PROMPT = """
Ты — **золотой копирайтер для «посева» рекомендаций**.
Пользователь даёт тебе **【название товара】**.

# ## 🎯 Ключевые правила
1.  **Боль клиента**: одной фразой объясни, зачем это покупать (лимитка? осветляет лицо? невероятная красота?).
2.  **Эмоциональная ценность**: используй живые слова вроде «вау», «атмосфера», «сердце замирает».
3.  **Лимит длины**: строго **до 40 символов**, коротко и ёмко.
4.  **Emoji**: обязательно 1–2 emoji.

# ## 🌟 Примеры (Few-Shot)
**Вход**: Dior 999 烈艳蓝金
**Выход**: 💄Королевский оттенок! Dior 999 — легендарный красный, осветляет лицо и добавляет статус. Идеально для подарка!

**Вход**: 泡泡玛特 Labubu 坐坐派对
**Выход**: ✨Слишком милота! Серия Labubu «坐坐派对» — каждая фигурка такая же, что хочется обнять. На столе — чистое умиление~

**Вход**: 罗技 MX Master 3S
**Выход**: 🖱️Лучший друг офисного работника! Logitech Master 3S — тихий и плавный, эргономика бережёт запястье.

---

**Текущая задача**:
Напиши одну фразу в стиле поста для соцсети про 【{input}】.
"""
pitcher_agent = SimpleAgent(
    llm=llm,
    name="Agent_Pitcher",
    system_prompt=PITCHER_PROMPT
)

print("✅ Агенты инициализированы!")


### Часть 4: Загрузка данных


In [ ]:
def load_user_profile(filename):
    # 1. Проверяем, существует ли файл
    if not os.path.exists(filename):
        print(f"⚠️ Файл конфигурации не найден: {filename}")
        # Если файла нет — записываем данные по умолчанию
        default_data = {
            "Пол": "жен.",
            "Возраст": "От 15 до 24 лет",
            "MBTI": "ENFP",
            "Созвездие": "Весы",
            "Бюджет:": "В пределах 500 юаней",
            "Фестиваль": "1-я годовщина любви",
            "Пользовательское": "Как и двумерные, обычно любят пить кофе, не присылают слишком практичные приборы"
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(default_data, f, ensure_ascii=False, indent=4)
        print(f"✅ Создан файл конфигурации по умолчанию. Измените {filename} и запустите снова.")
        return default_data

    # 2. Читаем содержимое файла
    try:
        with open(filename, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"✅ Профиль пользователя загружен: {filename}")
        print(f"📋 Предпросмотр: {json.dumps(data, ensure_ascii=False)}")
        return data
    except Exception as e:
        print(f"❌ Ошибка чтения JSON: {e}")
        return {}

# Загрузка данных
user_input_data = load_user_profile(INPUT_FILENAME)


### Часть 5: Генерация плана подарков


In [ ]:
def parse_budget_range(budget_str):
    """Разбирает строку бюджета пользователя, возвращает (min, max)"""
    nums = [float(x) for x in re.findall(r'\d+', str(budget_str).replace(',', ''))]
    if not nums: return 0, 999999 
    if "В пределах" in budget_str or "Общая фертильность" in budget_str: return 0, nums[0]
    if "и выше" in budget_str: return nums[0], 999999
    if len(nums) >= 2: return min(nums), max(nums)
    return 0, nums[0]

def extract_all_prices(raw_results):
    """Извлекает все валидные цены из списка результатов поиска"""
    prices = []
    for res in raw_results:
        # Обрабатываем только текстовые результаты
        if res.get('type') == 'text':
            text = res.get('title', '') + " " + res.get('content', '')
            # Ищем ¥, $,Юанейи т.п.
            matches = re.findall(r'(?:¥|￥|\$|HK\$|NT\$)\s*(\d+(?:,\d{3})*(?:\.\d+)?)', text)
            for m in matches:
                val = float(m.replace(',', ''))
                # Отфильтровываем годы (2025) и аномально малые/большие значения
                if 10 < val < 100000 and val not in [2024, 2025, 2026]:
                    prices.append(val)
            # Запасной regex:"ххх юаней"            matches_yuan = re.findall(r'(\ d + (?:,\ d {3}) * (?:\.\ d +)?)\ s * юань', text)
            for m in matches_yuan:
                val = float(m.replace(',', ''))
                if 10 < val < 100000 and val not in [2024, 2025, 2026]:
                    prices.append(val)
    return prices


def find_best_product(hunter, profiler_agent, keyword, budget_min, budget_max):
    limit_upper = budget_max * 1.2
    limit_lower = budget_min * 0.8
    
    all_candidates = []
    current_kw = keyword
    
    # --- Раунд 1: первый поиск ---
    print(f"       🕵️ 1-й поиск: {current_kw} цена")
    results_1 = hunter.search_raw(f"{current_kw} цена")
    
    fallback_img = ""
    for r in results_1:
        if r.get('images'): 
            fallback_img = r['images'][0]
            break

    has_valid_info = False
    for res in results_1:
        if res.get('type') == 'text':
            has_valid_info = True
            p_vals = extract_all_prices([res])
            if p_vals:
                res['price_val'] = p_vals[0]
                # 📝【Ключевое исправление 1】Запоминаем ключевое слово для результата
                res['source_kw'] = current_kw 
                all_candidates.append(res)
                
                if limit_lower <= p_vals[0] <= limit_upper:
                    if not res.get('img') and fallback_img: res['img'] = fallback_img
                    return res, f"около {p_vals[0]} юаней", current_kw

    # --- Механизм 3: защита при отсутствии данных ---
    if not has_valid_info:
        print(f"       ⚠️ [Механизм 3] Первый поиск не дал полезных данных.")
        correction_prompt = f"Стратегия '{current_kw}' не дала результатов. Предложи более популярную конкретную модель той же категории. Выведи только ключевое слово."
        new_kw = profiler_agent.run(correction_prompt).strip()
        print(f"       🔄 Стратег меняет запрос: {new_kw}")
        current_kw = new_kw
        
        results = hunter.search_raw(f"{current_kw} цена")
        
        # Обновляем fallback_img
        fallback_img = "" 
        for r in results:
            if r.get('images'): 
                fallback_img = r['images'][0]
                break
                
        for res in results:
            if res.get('type') == 'text':
                p_vals = extract_all_prices([res])
                if p_vals:
                    res['price_val'] = p_vals[0]
                    # 📝【Ключевое исправление 1】Запоминаем ключевое слово
                    res['source_kw'] = current_kw
                    all_candidates.append(res)

    # --- Механизмы 1 и 2: коррекция по цене ---
    avg_price = np.mean([c['price_val'] for c in all_candidates]) if all_candidates else 0
    
    if avg_price > 0:
        correction_prompt = ""
        if avg_price > limit_upper:
            print(f"       💸 [Механизм 1] Средняя цена {int(avg_price)} > верхняя граница {int(limit_upper)}, ищем аналог...")
            correction_prompt = f"Стратегия '{current_kw}' дала среднюю цену около {int(avg_price)} юаней, выше бюджета ({budget_max} юаней). Предложи более дешёвую конкретную модель той же категории (аналог). Выведи только ключевое слово."
        elif avg_price < limit_lower:
            print(f"       📉 [Механизм 2] Средняя цена {int(avg_price)} < нижняя граница {int(limit_lower)}, ищем премиум-версию...")
            correction_prompt = f"Стратегия '{current_kw}' дала среднюю цену около {int(avg_price)} юаней, ниже нижней границы бюджета ({budget_min} юаней). Предложи более дорогую модель той же категории. Выведи только ключевое слово."
            
        if correction_prompt:
            new_kw = profiler_agent.run(correction_prompt).strip()
            print(f"       🔄 Стратег корректирует: {new_kw}")
            current_kw = new_kw
            
            results_2 = hunter.search_raw(f"{new_kw} Цена")
            
            # Обновляем fallback_img
            fallback_img = "" 
            for r in results_2:
                if r.get('images'): 
                    fallback_img = r['images'][0]
                    break
            
            for res in results_2:
                if res.get('type') == 'text':
                    p_vals = extract_all_prices([res])
                    if p_vals:
                        res['price_val'] = p_vals[0]
                        # 📝【Ключевое исправление 1】Запоминаем ключевое слово
                        res['source_kw'] = current_kw
                        all_candidates.append(res) 
                        
                        if limit_lower <= p_vals[0] <= limit_upper:
                            if not res.get('img') and fallback_img: res['img'] = fallback_img
                            tag = "(аналог)" if avg_price > limit_upper else "(апгрейд)"
                            return res, f"около {p_vals[0]} юаней {tag}", current_kw

    # --- Механизм 4: резервная защита ---
    print("       ⚠️ [Механизм 4] Включён принудительный резервный режим...")
    best_fallback = None
    status_msg = "Цена не указана"
    
    if all_candidates:
        # Выбираем ближайший к бюджету
        target = (budget_min + budget_max) / 2
        best_fallback = sorted(all_candidates, key=lambda x: abs(x['price_val'] - target))[0]
        p = best_fallback['price_val']
        
        if p > limit_upper: status_msg = f"около {p} юаней (⚠️выше бюджета)"
        elif p < limit_lower: status_msg = f"около {p} юаней (📉ниже бюджета)"
        else: status_msg = f"около {p} юаней"
        
    elif results_1:
        # Если данных совсем нет — берём первый результат
        for res in results_1:
            if res.get('type') == 'text': 
                best_fallback = res
                # При резерве, если цены нет — используем исходное ключевое слово
                best_fallback['source_kw'] = keyword 
                break
    
    if best_fallback:
        if not best_fallback.get('img') and fallback_img:
            best_fallback['img'] = fallback_img
        
        # 📝【Ключевое исправление 2】Возвращаем source_kw из результата, а не current_kw
        final_name_to_use = best_fallback.get('source_kw', current_kw)
        
        return best_fallback, status_msg, final_name_to_use
        
    return None, "Поиск не удался", keyword

In [ ]:
if not user_input_data:
    print("❌ Данные пользователя не загружены")
else:
    # 0. Разбираем бюджет
    b_min, b_max = parse_budget_range(user_input_data.get('Бюджет:', ''))
    print(f"\n💰 Диапазон бюджета: {b_min} - {b_max} юаней")

    # 1. Стратег составляет план
    profile_text = "\n".join([f"- {k}: {v if v else 'неизвестно/без ограничений'}" for k, v in user_input_data.items()])
    print(f"\n🚀 Запуск задачи...\n{'-'*40}")
    print("\n🧠 [1/3] Стратег составляет первичный план...")
    search_strategy = profiler_agent.run(f"Составь стратегию поиска по профилю пользователя:\n\n{profile_text}")
    print(f"📝 Стратегия: \n{search_strategy}")

    # 2. Подготовка цикла
    keywords = [k.strip() for k in search_strategy.replace("，", ",").replace("\n", ",").split(',') if k.strip()]
    final_items = []
    hunter = BatchSearchTool()

    print(f"\n🔄 Обработка (всего {len(keywords)} товаров)...")

    for index, kw in enumerate(keywords):
        print(f"\n    👉 [Товар {index+1}/{len(keywords)}] Обработка: {kw}")
        
        # Вызываем умный поиск (передаём min и max)
        valid_result, price_status, final_kw = find_best_product(hunter, profiler_agent, kw, b_min, b_max)
        
        if not valid_result:
            print("       ❌ Данных нет, пропускаем.")
            continue
            
        # === Генерация текста ===
        product_name = valid_result.get('title', final_kw)
        
        print(f"       ✍️ Пишем текст: {product_name[:30]}...")
        pitch_prompt = f"""
        Товар: {product_name}
        Цена: {price_status}
        Фрагмент описания: {valid_result.get('content', '')[:200]}...
        
        Напиши одну рекомендательную фразу до 30 символов.
        """
        pitch = pitcher_agent.run(pitch_prompt)
        
        final_items.append({
            "name": final_kw, 
            "title_full": product_name,
            "price": price_status,
            "desc": pitch.replace("\n", " ").strip(),
            "img": valid_result.get('img', ''),
            "link": valid_result.get('url', '')
        })
        print(f"       ✅ Добавлено (статус: {price_status})")


### Часть 6: Вывод плана подарков


In [ ]:
# --- 4. Рендеринг и сохранение ---
print(f"\n💾 Формирование итогового отчёта...")

if not final_items:
    final_md = "К сожалению, при поиске в сети возникла проблема — информация о товарах не получена."
else:
    table_header = "| 🎁 Название подарка | 💰 Цена | ✨ Причина рекомендации | 🖼️ Изображение/ссылка |\n| :--- | :--- | :--- | :--- |\n"
    table_rows = []
    
    for item in final_items:
        # 1. Очищаем текстовые поля (символ | ломает таблицу)
        name = item.get('name', 'Неизвестно').replace("|", "/")
        price = item.get('price', 'Нет данных').replace("|", "/")
        desc = item.get('desc', '').replace("|", "/")
        
        # 2. 🚨【Ключевое исправление】Экранируем | в ссылках
        # Ссылки Baidu/JD часто содержат '|' — заменяем на '%7C', иначе таблица Markdown сломается
        raw_link = item.get('link', '#')
        safe_link = raw_link.replace("|", "%7C")
        
        raw_img = item.get('img', '')
        safe_img = raw_img.replace("|", "%7C")
        
        # 3. Формируем колонку с медиа
        if safe_img and safe_img.startswith("http"):
            # Картинка оборачивает ссылку на покупку
            media = f"[![Фото]({safe_img})]({safe_link})"
        else:
            media = f"[Купить]({safe_link})"
        
        # 4. Собираем строку (ссылка в названии тоже через safe_link)
        # strip() убирает возможные пробелы по краям
        row = f"| [{name}]({safe_link}) | {price} | {desc} | {media} |"
        table_rows.append(row)
        
    final_md = table_header + "\n".join(table_rows)
filename = "outputs/gift_plan_output.md"
# Создаём каталог вывода, если его нет
os.makedirs(os.path.dirname(filename), exist_ok=True)

with open(filename, "w", encoding="utf-8") as f:
    f.write(final_md)
print(f"🎉 Готово! Файл сохранён: {os.path.abspath(filename)}")

### Часть 7: Итоги и перспективы


#### Реализованные функции
- На основе профиля пользователя — рекомендации подарков в рамках бюджета
- Настраиваемый диапазон бюджета, праздник, личные предпочтения
- Два источника данных: Baidu MCP и Tavily API — актуальные цены и информация о товарах
- Визуальное представление результатов
#### Сложности и решения
- «Галлюцинации» LLM (ошибки JSON / выдуманные данные)
  - Решение: отказ от прямой генерации финальных данных моделью. Python-regex извлекает жёсткие данные (цена, картинка), LLM пишет только тексты. Точность — в коде, креатив — в модели.
- Слишком длинный контекст и сбои извлечения
  - Решение: ограничение длины ответа на этапе поиска; разделение «жёсткого потока» (параметры) и «мягкого потока» (продающие аргументы) — меньше контекста, выше скорость.
- Рекомендованные LLM подарки дороже бюджета
  - Решение: проверка цен. Если средняя цена выше бюджета, «стратег» пересматривает план и ищет «аналог», пока не найдёт подходящий товар.
- Проблемы формата параметров от Agent
  - Решение: на уровне инструмента — совместимость с разными форматами (JSON/строка, запятая/перенос строки), чтобы поисковые запросы не терялись.
#### Направления развития
- Фронтенд: веб-страница вместо Notebook для удобного взаимодействия
- Глубокая интеграция с Baidu Youxuan MCP: сравнение цен и история — точные цены, остатки, «сравнение по всей сети»
- Больше опций профиля: типы товаров, любимые бренды и т.д.
